# 🎨 محوّل الصور الفني - اليوم الوطني
# AI Style Transfer - National Day

---

## مرحباً بكم في تجربة التحويل الفني! 🇱🇾

### كيفية الاستخدام | How to Use:

1. **ارفع صورتك** أو التقط صورة بالكاميرا | Upload your photo or take a webcam shot
2. **اختر النمط الفني** المفضل لديك | Choose your preferred artistic style
3. **اضغط على زر التحويل** وانتظر النتيجة | Click Transform and wait for the result
4. **حمّل الصورة** الفنية الجديدة | Download your new artistic image

### الأنماط المتاحة | Available Styles:
- 🌌 **أنمي / كرتون** | Anime / Cartoon
- 🎨 **لوحة مائية** | Watercolor Painting
- 🖼️ **لوحة زيتية** | Oil Painting
- ✏️ **رسم بالرصاص** | Pencil Sketch
- 🇱🇾 **النمط الوطني** | National Day Theme

---
> **ملاحظة:** قد تستغرق عملية التحويل من 10 إلى 30 ثانية حسب النمط المختار
> **Note:** Processing may take 10-30 seconds depending on the chosen style

In [ ]:
# Cell 1: Environment Check + Install Maximum Quality Stack

import sys, subprocess, torch
print(f'Python {sys.version.split()[0]} | PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  ⚠ No GPU — Runtime > Change runtime type > T4 GPU')

print('\nInstalling maximum quality pipeline packages...')

# Core SD pipeline
subprocess.run(['pip','install','-q','diffusers>=0.27.0','transformers','accelerate','xformers'], check=False)

# Face identity: InsightFace (face embedding) + IP-Adapter FaceID
subprocess.run(['pip','install','-q','insightface','onnxruntime-gpu'], check=False)
subprocess.run(['pip','install','-q','git+https://github.com/tencent-ailab/IP-Adapter.git'], check=False)

# Post-processing: GFPGAN (face restoration) + Real-ESRGAN (4x upscale)
subprocess.run(['pip','install','-q','basicsr','facexlib','gfpgan','realesrgan'], check=False)

# UI
subprocess.run(['pip','install','-q','gradio'], check=False)

print('\n✓ All packages installed — proceed to Cell 2')

In [ ]:
# Cell 2: Imports & Global Setup

import os, gc, sys, time, warnings
import cv2, numpy as np, torch
import gradio as gr
from PIL import Image, ImageEnhance
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    StableDiffusionControlNetImg2ImgPipeline,
    StableDiffusionImg2ImgPipeline,
    DPMSolverMultistepScheduler,
)
from huggingface_hub import hf_hub_download
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32

print('=' * 60)
print('  Maximum Quality AI Style Transfer Pipeline')
print('  InsightFace → IP-Adapter FaceID → Counterfeit-V3')
print('  → GFPGAN face restore → Real-ESRGAN x2 upscale')
print('=' * 60)
print(f'Device : {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Global model handles (all lazy-loaded once)
_face_app        = None   # InsightFace — face embedding
_faceid_pipe     = None   # SD pipeline used by IPAdapterFaceID
_ip_faceid_model = None   # IPAdapterFaceID wrapper
_controlnet_pipe = None   # ControlNet img2img (fallback path)
_sd_pipe         = None   # Base SD 1.5 (watercolor/oil)
_restorer        = None   # GFPGAN
_upsampler       = None   # Real-ESRGAN anime
_USE_FACEID      = False  # set True after FaceID loads successfully

MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Cell 3: Download Weights & Load All Models

# ── 1. InsightFace face analyser ─────────────────────────
def load_face_app():
    global _face_app
    if _face_app: return _face_app
    print('[1/4] Loading InsightFace...')
    try:
        import insightface
        from insightface.app import FaceAnalysis
        _face_app = FaceAnalysis(
            name='buffalo_l',
            providers=['CUDAExecutionProvider','CPUExecutionProvider'])
        _face_app.prepare(ctx_id=0, det_size=(640,640))
        print('  InsightFace ready!')
    except Exception as e:
        print(f'  InsightFace failed: {e}')
        _face_app = None
    return _face_app


# ── 2. ONE pipeline: Counterfeit-V3.0 + ControlNet ───────
# Then we TRY to attach FaceID on top.
# If FaceID is unavailable, the pipeline still works standalone.
def load_main_pipeline():
    global _faceid_pipe, _ip_faceid_model, _USE_FACEID

    if _faceid_pipe:
        return _faceid_pipe

    print('[2/4] Loading ControlNet canny (~1.4 GB)...')
    cn = ControlNetModel.from_pretrained(
        'lllyasviel/sd-controlnet-canny', torch_dtype=DTYPE)

    # Anime models tried in order — first one that exists on HuggingFace is used
    ANIME_MODELS = [
        'dreamlike-art/dreamlike-anime-1.0',   # official, actively maintained
        'andite/anything-v4.0',                # popular anime model
        'runwayml/stable-diffusion-v1-5',      # base model (always exists)
    ]
    loaded = False
    for model_id in ANIME_MODELS:
        try:
            print(f'[2/4] Loading {model_id}...')
            _faceid_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
                model_id,
                controlnet=cn,
                torch_dtype=DTYPE,
                safety_checker=None,
                requires_safety_checker=False,
            )
            print(f'  Loaded: {model_id}')
            loaded = True
            break
        except Exception as e:
            print(f'  {model_id} failed ({type(e).__name__}) — trying next...')
    if not loaded:
        raise RuntimeError('All anime models failed to load')
    _faceid_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        _faceid_pipe.scheduler.config)
    _faceid_pipe = _faceid_pipe.to(DEVICE)

    if DEVICE == 'cuda':
        _faceid_pipe.enable_attention_slicing()
        _faceid_pipe.enable_vae_slicing()
        try: _faceid_pipe.enable_xformers_memory_efficient_attention()
        except: pass
        used  = torch.cuda.memory_allocated()/1e9
        total = torch.cuda.get_device_properties(0).total_memory/1e9
        print(f'  VRAM used: {used:.1f}/{total:.1f} GB')

    print('  Base pipeline ready! Trying FaceID...')

    # Download FaceID weights
    ckpt = f'{MODEL_DIR}/ip-adapter-faceid_sd15.bin'
    if not os.path.exists(ckpt):
        print('  Downloading FaceID weights (~100 MB)...')
        try:
            hf_hub_download('h94/IP-Adapter-FaceID',
                            'ip-adapter-faceid_sd15.bin',
                            local_dir=MODEL_DIR)
        except Exception as e:
            print(f'  FaceID download failed: {e}')

    if os.path.exists(ckpt):
        try:
            from ip_adapter.ip_adapter_faceid import IPAdapterFaceID
            _ip_faceid_model = IPAdapterFaceID(_faceid_pipe, ckpt, DEVICE)
            _USE_FACEID = True
            print('  IP-Adapter FaceID attached — face identity LOCKED!')
        except Exception as e:
            print(f'  FaceID attach failed: {e}')
            print('  Pipeline works WITHOUT FaceID (face identity not locked)')
            _USE_FACEID = False
    else:
        print('  FaceID weights missing — running without FaceID')
        _USE_FACEID = False

    return _faceid_pipe


# ── 3. GFPGAN face restorer ───────────────────────────────
def load_restorer():
    global _restorer
    if _restorer: return _restorer
    gfpgan_path = f'{MODEL_DIR}/GFPGANv1.4.pth'
    if not os.path.exists(gfpgan_path):
        print('[3/4] Downloading GFPGAN v1.4 (~330 MB)...')
        import urllib.request
        try:
            urllib.request.urlretrieve(
                'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
                gfpgan_path)
        except Exception as e:
            print(f'  GFPGAN download failed: {e}'); return None
    print('[3/4] Loading GFPGAN...')
    try:
        from gfpgan import GFPGANer
        _restorer = GFPGANer(
            model_path=gfpgan_path, upscale=1,
            arch='clean', channel_multiplier=2, bg_upsampler=None)
        print('  GFPGAN ready!')
    except Exception as e:
        print(f'  GFPGAN failed: {e}'); _restorer = None
    return _restorer


# ── 4. Real-ESRGAN anime upscaler ────────────────────────
def load_upsampler():
    global _upsampler
    if _upsampler: return _upsampler
    path = f'{MODEL_DIR}/RealESRGAN_x4plus_anime_6B.pth'
    if not os.path.exists(path):
        print('[4/4] Downloading Real-ESRGAN anime (~17 MB)...')
        import urllib.request
        try:
            urllib.request.urlretrieve(
                'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
                path)
        except Exception as e:
            print(f'  ESRGAN download failed: {e}'); return None
    print('[4/4] Loading Real-ESRGAN anime...')
    try:
        from realesrgan import RealESRGANer
        from basicsr.archs.rrdbnet_arch import RRDBNet
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_block=6, num_grow_ch=32, scale=4)
        _upsampler = RealESRGANer(
            scale=4, model_path=path, model=model,
            tile=512, tile_pad=10, pre_pad=0,
            half=(DEVICE == 'cuda'))
        print('  Real-ESRGAN ready!')
    except Exception as e:
        print(f'  Real-ESRGAN failed: {e}'); _upsampler = None
    return _upsampler


# ── Load everything ───────────────────────────────────────
print('Loading models — first run takes 5-10 min...\n')
load_face_app()
load_main_pipeline()
load_restorer()
load_upsampler()

print(f'\n{"="*55}')
print(f'Face embedding  : {"InsightFace + FaceID ✓" if _USE_FACEID else "basic (FaceID unavailable)"}')
print(f'Style model     : Counterfeit-V3.0 + ControlNet canny')
print(f'Face restore    : {"GFPGAN v1.4 ✓" if _restorer else "skipped"}')
print(f'Upscale         : {"Real-ESRGAN anime 2x ✓" if _upsampler else "skipped"}')
print(f'{"="*55}')

In [ ]:
# Cell 4: Style Transfer Functions

# ── Image helpers ─────────────────────────────────────────
def pil_to_cv2(img):
    return cv2.cvtColor(np.array(img.convert('RGB')), cv2.COLOR_RGB2BGR)

def cv2_to_pil(img):
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

def resize_to_sd(img, target=512):
    w, h = img.size
    nw, nh = (target, int(h*target/w)) if w>=h else (int(w*target/h), target)
    return img.resize((max(nw//8*8, 64), max(nh//8*8, 64)), Image.LANCZOS)

def resize_keep_aspect(img, max_side=768):
    w, h = img.size
    if max(w, h) <= max_side: return img
    s = max_side / max(w, h)
    return img.resize((int(w*s)//8*8, int(h*s)//8*8), Image.LANCZOS)

def get_canny(pil_image, low=60, high=160):
    img = np.array(pil_image.convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), low, high)
    return Image.fromarray(cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB))


# ── Seed & eye color helpers ────────────────────────────

def make_generator(seed=42):
    """Fixed seed → reproducible results. seed=-1 → random each time."""
    if seed == -1:
        seed = torch.randint(0, 2**32, (1,)).item()
    torch.manual_seed(seed)
    np.random.seed(seed % (2**31))
    return torch.Generator(device=DEVICE).manual_seed(seed), seed

def detect_eye_color(image):
    """
    Sample iris color at InsightFace eye keypoints.
    Returns a prompt-ready string like 'green eyes' or None if uncertain.
    """
    app = load_face_app()
    if app is None: return None
    try:
        img_array = np.array(image.convert('RGB'))
        faces = app.get(img_array)
        if not faces: return None
        face  = max(faces, key=lambda f: f.det_score)
        kps   = face.kps          # [left_eye, right_eye, nose, ...]
        h, w  = img_array.shape[:2]
        r     = max(4, min(h, w) // 55)   # adaptive iris sample radius

        patches = []
        for eye_pt in kps[:2]:
            x, y = int(eye_pt[0]), int(eye_pt[1])
            patch = img_array[max(0,y-r):y+r, max(0,x-r):x+r]
            if patch.size: patches.append(patch.reshape(-1, 3).mean(0))

        if not patches: return None
        rv, gv, bv = np.mean(patches, axis=0)

        # Heuristic classification (works well for portraits)
        if bv > max(rv, gv) + 18:
            return 'blue eyes'
        if gv > max(rv, bv) + 12:
            return 'green eyes'
        if rv > bv + 25 and gv > bv + 10:
            return 'hazel eyes' if rv > 140 else 'brown eyes'
        return 'dark brown eyes'
    except Exception as e:
        print(f'  Eye color detection error: {e}'); return None

# ── Face embedding ────────────────────────────────────────
def get_face_embedding(image):
    app = load_face_app()
    if app is None: return None
    try:
        faces = app.get(np.array(image.convert('RGB')))
        if not faces: return None
        best = max(faces, key=lambda f: f.det_score)
        return torch.from_numpy(best.normed_embedding).unsqueeze(0)
    except Exception as e:
        print(f'  Face embedding error: {e}'); return None


# ── Post-processing ───────────────────────────────────────
def restore_face(pil_image):
    r = load_restorer()
    if r is None: return pil_image
    try:
        _, _, restored = r.enhance(
            pil_to_cv2(pil_image),
            has_aligned=False, only_center_face=False, paste_back=True)
        return cv2_to_pil(restored)
    except Exception as e:
        print(f'  GFPGAN error: {e}'); return pil_image

def upscale_2x(pil_image):
    u = load_upsampler()
    if u is None: return pil_image
    try:
        output, _ = u.enhance(pil_to_cv2(pil_image), outscale=2)
        return cv2_to_pil(output)
    except Exception as e:
        print(f'  ESRGAN error: {e}'); return pil_image


# ── OpenCV guaranteed fallback ────────────────────────────
def _anime_opencv(image):
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h, w) > 800:
        s = 800/max(h,w); img = cv2.resize(img,(int(w*s)//2*2,int(h*s)//2*2))
    smooth = img.copy()
    for _ in range(8): smooth = cv2.bilateralFilter(smooth, 9, 75, 75)
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray,7), 255,
                cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 2)
    res = cv2.bitwise_and(smooth, cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR))
    hsv = cv2.cvtColor(res, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1]*2.3, 0, 255)
    hsv[:,:,2] = np.clip(hsv[:,:,2]*1.1, 0, 255)
    out = cv2_to_pil(cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR))
    return ImageEnhance.Color(out.resize(image.size, Image.LANCZOS)).enhance(1.6)


# ── Prompts per style ─────────────────────────────────────
PROMPTS = {
    'kawaii': (
        "(masterpiece:1.4),(best quality:1.3),(ultra-detailed:1.2),"
        "kawaii anime girl,chibi style,big sparkling eyes,"
        "soft round face,smooth cel shading,pastel colors,warm lighting",
        "(worst quality:1.4),(low quality:1.4),realistic,photo,"
        "ugly,bad anatomy,deformed,extra limbs,text,watermark,nsfw"
    ),
    'anime_hd': (
        "(masterpiece:1.3),(best quality:1.2),"
        "anime portrait,beautiful detailed eyes,clean sharp lineart,"
        "vibrant colors,cel shaded,professional anime illustration",
        "(worst quality:1.4),(low quality:1.4),realistic,photo,"
        "blurry,bad anatomy,deformed,text,watermark,nsfw"
    ),
    'cartoon': (
        "(masterpiece:1.3),(best quality:1.2),"
        "studio ghibli style,cartoon character,smooth rounded face,"
        "big expressive eyes,warm soft lighting,vibrant flat colors",
        "(worst quality:1.4),(low quality:1.4),realistic,photo,"
        "blurry,bad anatomy,text,watermark,dark,nsfw"
    ),
    'cartoon_portrait': (
        "(masterpiece:1.4),(best quality:1.3),(ultra-detailed:1.2),"
        "western cartoon portrait,disney style,cartoon character,"
        "thick black outlines,big expressive eyes,detailed curly hair,"
        "smooth clean skin,vivid saturated colors,2D flat illustration,"
        "professional cartoon art,full body,same outfit as original",
        "(worst quality:1.4),(low quality:1.4),anime,chibi,realistic,photo,"
        "3d render,ugly,bad anatomy,deformed,text,watermark,nsfw,blurry"
    ),
}


# ── Core generator — 3-level fallback chain ───────────────
def generate_anime(image, style_key='kawaii',
                    steps=30, guidance=9.0, cn_scale=0.70,
                    faceid_scale=0.8, img2img_strength=0.60,
                    seed=42):
    """
    Fallback chain — guarantees a result:
      A) img2img + ControlNet + FaceID  (max quality + identity preserved)
      B) img2img + ControlNet           (reliable identity via pixel content)
      C) AnimeGANv2 → OpenCV            (always works)

    seed: fixed → same result every run. -1 → different each time.
    Eye color: auto-detected from photo, injected into prompt.
    """
    img_512     = resize_to_sd(image.convert('RGB'), 512)
    prompt, neg = PROMPTS.get(style_key, PROMPTS['kawaii'])
    canny       = get_canny(img_512, 60, 160)
    generator, used_seed = make_generator(seed)
    print(f'  Seed: {used_seed}')

    # ── Auto-detect eye color → inject into prompt ────────
    eye_color = detect_eye_color(img_512)
    if eye_color:
        print(f'  Eye color detected: {eye_color}')
        # Force correct eye color; push wrong colors to negative
        prompt = f'({eye_color}:1.5), ' + prompt
        wrong  = {'blue eyes','green eyes','brown eyes',
                  'hazel eyes','dark brown eyes'} - {eye_color}
        neg    = ', '.join(wrong) + ', ' + neg
    else:
        print('  Eye color: not detected (face not found)')

    ctx = torch.autocast(DEVICE) if DEVICE == 'cuda' else torch.no_grad()
    result = None

    # ── Level A: img2img + ControlNet + FaceID ────────────
    if _USE_FACEID and _ip_faceid_model is not None:
        print('  [A] img2img + ControlNet + FaceID...')
        try:
            faceid_embeds = get_face_embedding(img_512)
            if faceid_embeds is None:
                faceid_embeds = torch.zeros(1, 512)
            with ctx:
                images = _ip_faceid_model.generate(
                    faceid_embeds=faceid_embeds,
                    prompt=prompt, negative_prompt=neg,
                    scale=faceid_scale,
                    num_samples=1,
                    num_inference_steps=steps,
                    guidance_scale=guidance,
                    seed=used_seed,            # reproducible
                    image=img_512,
                    control_image=canny,
                    strength=img2img_strength,
                    controlnet_conditioning_scale=cn_scale,
                )
            result = images[0]
            print('  [A] Success!')
        except Exception as e:
            print(f'  [A] FaceID error: {e}')

    # ── Level B: img2img + ControlNet (no FaceID) ─────────
    # Starts from the real photo → same person's face features preserved
    if result is None and _faceid_pipe is not None:
        print('  [B] img2img + ControlNet (face from photo pixels)...')
        try:
            with ctx:
                result = _faceid_pipe(
                    prompt=prompt, negative_prompt=neg,
                    image=img_512,
                    control_image=canny,
                    strength=img2img_strength,
                    guidance_scale=guidance,
                    num_inference_steps=steps,
                    controlnet_conditioning_scale=cn_scale,
                    generator=generator,       # reproducible
                ).images[0]
            print('  [B] Success!')
        except Exception as e:
            print(f'  [B] img2img error: {e}')

    # ── Level C: AnimeGANv2 → OpenCV ──────────────────────
    if result is None:
        print('  [C] AnimeGAN/OpenCV guaranteed fallback...')
        result = anime_fast_style(image)

    if DEVICE == 'cuda':
        torch.cuda.empty_cache(); gc.collect()

    # ── Post-processing: GFPGAN face restore + ESRGAN 2x ──
    print('  Post: GFPGAN face restore + Real-ESRGAN 2x upscale...')
    result = restore_face(result)
    result = upscale_2x(result)
    return result


# ── Style wrappers ────────────────────────────────────────
def kawaii_anime_style(image, strength=0.7, seed=42):
    return generate_anime(image, 'kawaii',
                          steps=30, guidance=9.0, cn_scale=0.70,
                          faceid_scale=0.85, img2img_strength=0.60, seed=seed)

def anime_hd_style(image, strength=0.65, seed=42):
    return generate_anime(image, 'anime_hd',
                          steps=28, guidance=8.5, cn_scale=0.65,
                          faceid_scale=0.80, img2img_strength=0.58, seed=seed)

def cartoon_style(image, strength=0.68, seed=42):
    return generate_anime(image, 'cartoon',
                          steps=28, guidance=9.0, cn_scale=0.75,
                          faceid_scale=0.75, img2img_strength=0.65, seed=seed)

def cartoon_portrait_style(image, strength=0.72, seed=42):
    """Western cartoon / Disney style — closest to @whincy_lab aesthetic."""
    return generate_anime(image, 'cartoon_portrait',
                          steps=32, guidance=9.5, cn_scale=0.65,
                          faceid_scale=0.90, img2img_strength=0.62, seed=seed)


# ── OpenCV-only styles (instant, no GPU model) ────────────
def watercolor_style(image, strength=0.6):
    img = pil_to_cv2(image)
    h,w = img.shape[:2]
    if max(h,w)>1024: s=1024/max(h,w); img=cv2.resize(img,(int(w*s),int(h*s)))
    out = cv2.stylization(img,sigma_s=int(60+strength*60),sigma_r=0.3+strength*0.25)
    out = cv2.addWeighted(out,0.7,cv2.GaussianBlur(out,(3,3),0),0.3,0)
    hsv = cv2.cvtColor(out,cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1]*1.3,0,255)
    return ImageEnhance.Color(cv2_to_pil(
        cv2.cvtColor(hsv.astype(np.uint8),cv2.COLOR_HSV2BGR))).enhance(1.15)

def oil_painting_style(image, strength=0.6):
    img = pil_to_cv2(image)
    h,w = img.shape[:2]
    if max(h,w)>1024: s=1024/max(h,w); img=cv2.resize(img,(int(w*s),int(h*s)))
    out = cv2.detailEnhance(img,sigma_s=int(10+strength*10),sigma_r=0.15)
    n = max(6,int(16-strength*8))
    _,lbl,ctr = cv2.kmeans(np.float32(out).reshape((-1,3)),n,None,
        (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,20,0.5),5,
        cv2.KMEANS_RANDOM_CENTERS)
    quant = np.uint8(ctr)[lbl.flatten()].reshape(out.shape)
    blend = cv2.edgePreservingFilter(
        cv2.addWeighted(out,0.4,quant,0.6,0),flags=1,sigma_s=30,sigma_r=0.4)
    return ImageEnhance.Color(
        ImageEnhance.Contrast(cv2_to_pil(blend)).enhance(1.1)).enhance(1.2)

def pencil_sketch(image, strength=0.6):
    img = pil_to_cv2(image)
    h,w = img.shape[:2]
    if max(h,w)>1024: s=1024/max(h,w); img=cv2.resize(img,(int(w*s),int(h*s)))
    gray,_ = cv2.pencilSketch(img,sigma_s=60,sigma_r=0.07,
                               shade_factor=0.03+strength*0.05)
    return ImageEnhance.Contrast(
        cv2_to_pil(cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR))).enhance(1.3)

_animegan_model = None
def anime_fast_style(image, strength=0.55):
    global _animegan_model
    try:
        from torchvision.transforms.functional import to_tensor, to_pil_image
        if _animegan_model is None:
            _animegan_model = torch.hub.load(
                'bryandlee/animegan2-pytorch:main','generator',
                pretrained='face_paint_512_v2',force_reload=False
            ).to(DEVICE).eval()
        w,h = image.size; sq = min(w,h,512)
        crop = image.crop(((w-sq)//2,(h-sq)//2,(w+sq)//2,(h+sq)//2))
        crop = crop.resize((512,512),Image.LANCZOS)
        x = (to_tensor(crop).unsqueeze(0)*2-1).to(DEVICE)
        with torch.no_grad(): out = _animegan_model(x)
        res = to_pil_image((out.squeeze(0)*0.5+0.5).clamp(0,1))
        return ImageEnhance.Color(res).enhance(1.9).resize(image.size,Image.LANCZOS)
    except Exception as e:
        print(f'  AnimeGAN error: {e}')
    return _anime_opencv(image)


# ── Style map ─────────────────────────────────────────────
STYLE_MAP = {
    'Kawaii Anime / أنمي كاواي':    ('kawaii',   kawaii_anime_style),
    'Anime HD / انمي عالي الجودة':  ('anime_hd', anime_hd_style),
    'Anime Fast (AnimeGAN)':         ('anime_f',  anime_fast_style),
    'Cartoon Portrait / كرتون غربي':   ('cartoon_p', cartoon_portrait_style),
    'Cartoon Ghibli / كرتون غيبلي':   ('cartoon',   cartoon_style),
    'Watercolor / الوان مائية':      ('water',    watercolor_style),
    'Oil Painting / زيتية':          ('oil',      oil_painting_style),
    'Pencil Sketch / رصاص':          ('sketch',   pencil_sketch),
}


def transform_image(image, style_name, strength=0.55, seed=42):
    if image is None:
        return None, 'Upload an image | ارفع صورة'
    try:
        pil = (Image.fromarray(image) if isinstance(image,np.ndarray)
               else image).convert('RGB')
        pil = resize_keep_aspect(pil, 768)
        _,fn = STYLE_MAP.get(style_name, ('kawaii', kawaii_anime_style))
        seed = int(seed)
        print(f'\nGenerating: {style_name} | seed={seed}')
        t0 = time.time()
        result = fn(pil, strength=strength, seed=seed)
        elapsed = time.time()-t0
        path = ('FaceID+ControlNet' if _USE_FACEID else 'ControlNet') + '+GFPGAN+ESRGAN'
        return result, f'Done {elapsed:.1f}s | {path} | {result.width}×{result.height}px'
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        return None, 'GPU OOM — Runtime > Restart & run all cells'
    except Exception as e:
        import traceback
        return None, f'Error: {traceback.format_exc()[-400:]}'


print('Style functions ready.')
print(f'Pipeline: {"FaceID (max quality)" if _USE_FACEID else "ControlNet fallback"}')
print('Fallback chain: FaceID → ControlNet → AnimeGAN → OpenCV')

In [ ]:
# Cell 6: Gradio UI (Webcam + Styles)

import gradio as gr

STYLES = list(STYLE_MAP.keys())

with gr.Blocks(
    title='AI Style Transfer — Libya Tech & IT Day',
    theme=gr.themes.Soft(primary_hue='blue'),
) as demo:

    gr.Markdown('# AI Style Transfer')
    gr.Markdown('## Libya Tech & IT Day | يوم التقنية والمعلومات - ليبيا')

    # ── Input section ─────────────────────────────────────
    gr.Markdown('### Step 1: Get your photo | الخطوة 1: احصل على صورتك')

    with gr.Tabs():
        with gr.Tab('Webcam (Selfie) | كاميرا'):
            webcam_input = gr.Image(
                sources=['webcam'],
                type='pil',
                label='Take a photo with your laptop camera | التقط صورة بكاميرا اللابتوب',
                mirror_webcam=True,
                height=320,
            )
            use_webcam_btn = gr.Button('Use this photo | استخدم هذه الصورة', variant='secondary')

        with gr.Tab('Upload | رفع صورة'):
            upload_input = gr.Image(
                sources=['upload', 'clipboard'],
                type='pil',
                label='Upload from device | ارفع من الجهاز',
                height=320,
            )

    # Shared working image (fed from either tab)
    working_image = gr.Image(
        type='pil',
        label='Selected photo | الصورة المختارة',
        height=200,
        interactive=False,
    )

    # Copy webcam capture to working image
    use_webcam_btn.click(fn=lambda x: x,
                         inputs=[webcam_input], outputs=[working_image])
    upload_input.change(fn=lambda x: x,
                        inputs=[upload_input], outputs=[working_image])

    # ── Style + transform ─────────────────────────────────
    gr.Markdown('### Step 2: Choose style & transform | الخطوة 2: اختر النمط وحول')

    with gr.Row():
        with gr.Column(scale=1):
            style_selector = gr.Radio(
                choices=STYLES,
                value=STYLES[0],
                label='Art Style | النمط الفني',
            )
            strength_slider = gr.Slider(
                minimum=0.30, maximum=0.80, value=0.55, step=0.05,
                label='Style Strength (Cartoon / Anime HD) | قوة الاسلوب',
            )
            seed_input = gr.Number(
                value=42, precision=0,
                label='Seed | البذرة  (42=ثابت دائماً،  -1=عشوائي كل مرة)',
            )
            transform_btn = gr.Button(
                'Transform | حول الان',
                variant='primary',
                size='lg',
            )

        with gr.Column(scale=1):
            output_image = gr.Image(
                label='Result | النتيجة',
                type='pil',
                height=320,
                interactive=False,
            )
            status_box = gr.Textbox(
                label='Status | الحالة',
                lines=2,
                interactive=False,
            )
            download_btn = gr.DownloadButton(
                label='Download | تحميل',
                visible=False,
            )

    gr.Markdown(
        '**Speed:** Anime Fast ~5s  |  Anime HD / Cartoon-Pixar ~30s (SD)  |  Others ~3s'
    )

    def on_transform(image, style, strength, seed=42):
        result, status = transform_image(image, style, strength, seed=int(seed))
        if result is not None:
            tmp = '/tmp/styled_image.png'
            result.save(tmp)
            return result, status, gr.update(value=tmp, visible=True)
        return None, status, gr.update(visible=False)

    transform_btn.click(
        fn=on_transform,
        inputs=[working_image, style_selector, strength_slider, seed_input],
        outputs=[output_image, status_box, download_btn],
    )

print('Launching...')
demo.queue(max_size=5).launch(share=True, debug=False, show_error=True)


In [ ]:
# ============================================================
# Cell 7: Wireless Printing | Tabia Lasilkiya
# ============================================================
# Adds a branded National Day frame and opens the browser print
# dialog so the image is sent to any connected WiFi printer.
# ============================================================

import base64, io, datetime
from PIL import Image, ImageDraw, ImageFont
import gradio as gr


# -- A: Branded print card ---------------------------------

def create_print_card(
    image,
    title_ar='يوم التقنية والمعلومات - ليبيا',
    title_en='Libya Tech & IT Day',
    subtitle='AI Style Transfer . تحويل الصور بالذكاء الاصطناعي',
    card_size=(1200, 900),
):
    """Wrap the styled image in a branded National Day print card."""
    W, H    = card_size
    BORDER  = 30
    HEADER  = 80
    FOOTER  = 55
    PADDING = 12
    GREEN   = (10, 22, 40)
    GOLD    = (0, 168, 255)

    canvas = Image.new('RGB', (W, H), color=GREEN)

    ix0, iy0 = BORDER, BORDER + HEADER
    ix1, iy1 = W - BORDER, H - BORDER - FOOTER
    canvas.paste(Image.new('RGB', (ix1-ix0, iy1-iy0), 'white'), (ix0, iy0))

    inner_w = ix1 - ix0 - 2*PADDING
    inner_h = iy1 - iy0 - 2*PADDING
    thumb = image.copy().convert('RGB')
    thumb.thumbnail((inner_w, inner_h), Image.LANCZOS)
    px = ix0 + PADDING + (inner_w - thumb.width)  // 2
    py = iy0 + PADDING + (inner_h - thumb.height) // 2
    canvas.paste(thumb, (px, py))

    draw = ImageDraw.Draw(canvas)

    draw.rectangle([BORDER//2, BORDER//2, W-BORDER//2, H-BORDER//2],
                   outline=GOLD, width=3)
    draw.rectangle([BORDER//2+6, BORDER//2+6, W-BORDER//2-6, H-BORDER//2-6],
                   outline=GOLD, width=1)

    def font(px):
        for path in [
            '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
            '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
            '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf',
        ]:
            try:
                return ImageFont.truetype(path, px)
            except Exception:
                pass
        return ImageFont.load_default()

    draw.text((W//2, BORDER + HEADER//2),
              f'{title_ar}  |  {title_en}',
              fill=GOLD, font=font(30), anchor='mm')

    draw.text((W//2, H - BORDER - FOOTER//2 - 10),
              subtitle,
              fill=(180, 220, 180), font=font(19), anchor='mm')
    draw.text((W//2, H - BORDER - FOOTER//2 + 14),
              datetime.datetime.now().strftime('%Y/%m/%d'),
              fill=(150, 190, 160), font=font(16), anchor='mm')

    sq = 14
    for cx, cy in [(BORDER+4, BORDER+4), (W-BORDER-4-sq, BORDER+4),
                   (BORDER+4, H-BORDER-4-sq), (W-BORDER-4-sq, H-BORDER-4-sq)]:
        draw.rectangle([cx, cy, cx+sq, cy+sq], fill=GOLD)

    return canvas


# -- B: Auto-print HTML builder ---------------------------

def build_print_html(image):
    """Encode image as base64 and return HTML with auto-print JavaScript."""
    if image is None:
        return "<p style='color:#f88;'>Upload image first</p>"

    buf = io.BytesIO()
    image.save(buf, format='PNG', dpi=(300, 300))
    b64 = base64.b64encode(buf.getvalue()).decode()

    html_page = (
        '<!DOCTYPE html><html><head><title>National Day Print</title>'
        '<style>'
        'body{margin:0;padding:0;background:#fff;display:flex;'
        'justify-content:center;align-items:center;min-height:100vh;}'
        'img{max-width:100%;max-height:97vh;object-fit:contain;}'
        '@media print{img{width:100%;height:auto;page-break-inside:avoid;}}'
        '</style></head><body>'
        f'<img src="data:image/png;base64,{b64}"'
        ' onload="setTimeout(()=>{window.print();setTimeout(()=>window.close(),2500);},400)">'
        '</body></html>'
    )

    # Escape for JavaScript string
    js_page = html_page.replace('`', r'\`').replace('${', r'\${')

    return (
        '<div style="text-align:center;padding:14px;">'
        '<button onclick="openPrint()" style="'
        'background:linear-gradient(135deg,#006C35,#00A550);'
        'color:white;border:2px solid #C8A951;'
        'padding:14px 38px;font-size:1.15em;font-weight:bold;'
        'border-radius:10px;cursor:pointer;">'
        '&#128424; &nbsp; '
        '\u0637\u0628\u0627\u0639\u0629 \u0639\u0644\u0649 \u0627\u0644\u0637\u0627\u0628\u0639\u0629 \u0627\u0644\u0644\u0627\u0633\u0644\u0643\u064a\u0629'
        ' &nbsp;&middot;&nbsp; Print to Wireless Printer'
        '</button>'
        '<p style="color:#aaa;font-size:0.82em;margin:6px 0 0;">'
        '\u0633\u062a\u0641\u062a\u062d \u0646\u0627\u0641\u0630\u0629 \u0627\u0644\u0637\u0627\u0628\u0639\u0629 \u062a\u0644\u0642\u0627\u0626\u064a\u0627\u064b'
        ' &nbsp;&middot;&nbsp; Print dialog opens automatically'
        '</p></div>'
        f'<script>function openPrint(){{var w=window.open("","_blank","width=980,height=740");w.document.write(`{js_page}`);w.document.close();}}\n</script>'
    )


# -- C: Printing UI ---------------------------------------

print_css = 'body, .gradio-container { background: linear-gradient(135deg, #0A2B1A, #0D3B22) !important; }'

with gr.Blocks(css=print_css, title='Print | Tiba3a') as print_demo:

    gr.HTML("""
    <div style='text-align:center;padding:20px 0 8px;'>
      <span style='color:#C8A951;font-size:1.7em;font-weight:bold;'>
        &#128424; \u0637\u0628\u0627\u0639\u0629 \u0627\u0644\u0635\u0648\u0631\u0629 \u0627\u0644\u0641\u0646\u064a\u0629 &nbsp;&middot;&nbsp; Print Your Art
      </span><br>
      <span style='color:#8FD4A8;font-size:0.95em;'>
        \u0627\u0644\u064a\u0648\u0645 \u0627\u0644\u0648\u0637\u0646\u064a \u0627\u0644\u0633\u0639\u0648\u062f\u064a &#127480;&#127462;
      </span>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            print_input = gr.Image(
                label='Image to Print',
                type='pil',
                sources=['upload', 'clipboard'],
                height=310,
            )
            add_frame = gr.Checkbox(
                label='Add National Day Frame (green + gold)',
                value=True,
            )
            print_btn = gr.Button(
                '&#128424; Print Now',
                variant='primary',
            )

        with gr.Column(scale=1):
            preview = gr.Image(
                label='Print Preview',
                type='pil',
                height=310,
                interactive=False,
            )
            print_html = gr.HTML()

    def on_print(img, frame):
        if img is None:
            return None, "<p style='color:#f88;text-align:center;'>Upload image first</p>"
        card = create_print_card(img) if frame else img
        return card, build_print_html(card)

    print_btn.click(
        fn=on_print,
        inputs=[print_input, add_frame],
        outputs=[preview, print_html],
    )

    gr.HTML("""
    <div style='background:rgba(200,169,81,0.08);border:1px solid rgba(200,169,81,0.3);
                border-radius:10px;padding:14px;margin-top:12px;
                color:#C8A951;font-size:0.88em;line-height:1.8;'>
      <strong>Steps / \u062e\u0637\u0648\u0627\u062a:</strong><br>
      1. Upload the styled image (or paste from clipboard)<br>
      2. Enable 'National Day Frame' for green+gold border<br>
      3. Click Print Now -- print dialog opens automatically<br>
      4. Select your wireless printer and print!<br><br>
      <strong>Tip:</strong> Printer must be on same WiFi with driver installed
    </div>
    """)

print('=' * 60)
print('Wireless Printing Interface ready!')
print('=' * 60)
print_demo.launch(share=True, quiet=True)


# 💡 Tips & Troubleshooting | نصائح واستكشاف الأخطاء

---

## ⚡ Performance Tips | نصائح الأداء

### Getting the Best Results | للحصول على أفضل النتائج:

| Style | Best Input | Strength | Time |
|-------|-----------|----------|------|
| 🌌 Anime | Portrait, face photo | 0.5-0.65 | ~25s |
| 🎨 Watercolor | Landscape, colorful | 0.5-0.7 | ~3s |
| 🖼️ Oil Painting | Portrait, any photo | 0.5-0.7 | ~5s |
| ✏️ Pencil Sketch | Any photo | 0.3-0.6 | ~2s |
| 🇱🇾 National Day | Portrait, group | 0.5-0.65 | ~5s |

---

## 🔧 Common Issues | المشكلات الشائعة

### ❌ "GPU out of memory" Error
```
Solution:
1. Runtime > Restart runtime
2. Re-run all cells
3. Use a smaller image (< 512px)
4. Reduce Style Strength slider
```

### ❌ Anime style is slow or fails
```
Possible causes:
• Model still downloading (first run takes 2-5 min)
• GPU memory full → restart runtime
• No GPU selected → Runtime > Change runtime type > T4

Solutions:
• Wait for download to complete
• Use watercolor/sketch while waiting
• Restart and re-run cells
```

### ❌ "No GPU detected" warning
```
Steps to enable GPU:
1. Runtime → Change runtime type
2. Hardware accelerator → T4 GPU
3. Save → Disconnect and Reconnect
4. Re-run all cells from top
```

### ❌ Gradio link expired
```
Free Gradio share links expire after 72 hours.
Re-run Cell 6 to get a new link.
```

---

## 📱 For Event Staff | لموظفي الفعالية

### Setup Checklist:
- [ ] Open notebook in Google Colab
- [ ] Enable T4 GPU (Runtime > Change runtime type)
- [ ] Run all cells in order (Runtime > Run all)
- [ ] Wait for the public Gradio URL to appear
- [ ] Share the URL or QR code with visitors
- [ ] Keep the Colab tab open (do not close!)

### Visitor Instructions (Arabic):
```
مرحباً بك في تجربة التحويل الفني! 🎨
1. ارفع صورتك أو اسحبها
2. اختر النمط الفني المفضل
3. اضغط "حوّل الآن"
4. انتظر 5-30 ثانية
5. حمّل الصورة الفنية الجديدة!
```

### Quick Reset if Something Goes Wrong:
1. Runtime → Restart and Run All
2. Wait ~5 minutes for models to reload
3. Share the new Gradio URL with visitors

---

## 🌟 Advanced Usage | الاستخدام المتقدم

### Customizing Prompts (Anime Style):
To modify the anime style prompt, edit Cell 5, `anime_style()` function:
```python
prompt = "YOUR CUSTOM PROMPT, anime style, ..."
negative_prompt = "realistic, photo, blurry, ..."
```

### Adding New Styles:
1. Add a new function in Cell 5 following the same pattern
2. Add it to the `STYLE_MAP` dictionary
3. Re-run Cell 5 and Cell 6

### Batch Processing:
For multiple images, call `transform_image()` directly:
```python
from PIL import Image
img = Image.open("your_photo.jpg")
result, status = transform_image(img, "🌌 Anime / أنمي", strength=0.6)
result.save("output.png")
```

---

## 📞 Technical Specs | المواصفات التقنية

- **Base Model**: Stable Diffusion v1.5 (runwayml/stable-diffusion-v1-5)
- **CV Styles**: OpenCV 4.8+ (watercolor, oil, sketch, national day)
- **GPU**: Google Colab T4 (16 GB VRAM)
- **Framework**: Gradio 3.50 + Diffusers 0.21
- **Target**: < 30 seconds per image
- **Max image size**: 768×768 px (auto-resized)

---

*🇱🇾 Developed for Libya Tech & IT Day Event · تطوير لفعالية يوم التقنية والمعلومات - ليبيا*